# 📊 Análisis con MLlib en PySpark  
## Predicción de tarifa en NYC Yellow Taxi (2025)

---

## 1️⃣ Objetivo

Realizar un análisis de Machine Learning utilizando MLlib de PySpark sobre el conjunto de datos NYC Yellow Taxi (enero–marzo 2025), con el objetivo de predecir la variable `fare_amount`.

---

## 2️⃣ Preparación de datos

Se cargaron múltiples archivos en formato Parquet y se realizó un proceso de limpieza:

- Eliminación de valores NULL en variables relevantes.
- Eliminación de valores inválidos (distancia y tarifa menores o iguales a 0).
- Eliminación de valores atípicos extremos:
  - `fare_amount < 200`
  - `trip_distance < 100`
  - `tip_amount < 200`

El filtrado de outliers fue necesario debido a la presencia de valores extremos que afectaban negativamente el rendimiento del modelo.

---

## 3️⃣ Variables utilizadas

Se utilizaron las siguientes variables como predictores:

- `trip_distance`
- `passenger_count`
- `payment_type`
- `tip_amount`
- `tolls_amount`
- `extra`
- `congestion_surcharge`

La variable objetivo fue:

- `fare_amount`

Se utilizó `VectorAssembler` para construir el vector de características requerido por MLlib.

---

## 4️⃣ Modelo implementado

Se entrenó un modelo:

### 🌲 RandomForestRegressor

Este modelo fue seleccionado debido a su capacidad para:

- Capturar relaciones no lineales
- Manejar grandes volúmenes de datos
- Reducir el riesgo de sobreajuste
- Proporcionar interpretabilidad mediante importancia de variables

Se dividieron los datos en:

- 80% entrenamiento
- 20% prueba

---

## 5️⃣ Resultados del modelo

Métricas obtenidas:

- **RMSE:** 4.50  
- **R²:** 0.924  

### Interpretación:

- El modelo explica aproximadamente el **92.4% de la variabilidad** de la tarifa.
- El error promedio de predicción es de aproximadamente **4.5 dólares**, lo cual es razonable considerando la variabilidad natural del servicio de taxi.

---

## 6️⃣ Importancia de variables

Las variables con mayor influencia en la predicción fueron:

1. `trip_distance` (~54%)
2. `tip_amount` (~24%)
3. `tolls_amount` (~16%)

Variables como `payment_type` y `passenger_count` mostraron impacto casi nulo, lo cual es consistente con la lógica del negocio, ya que no determinan directamente la tarifa base.

---

## 7️⃣ Conclusión

El uso de MLlib en PySpark permitió realizar un análisis predictivo eficiente sobre un conjunto de datos masivo.

Inicialmente, el modelo presentaba bajo desempeño debido a la presencia de valores atípicos extremos. Tras aplicar un filtrado adecuado, el modelo Random Forest mostró una alta capacidad predictiva.

Este ejercicio demuestra:

- La importancia del preprocesamiento de datos.
- La utilidad de modelos no lineales en problemas reales.
- La capacidad de MLlib para trabajar con grandes volúmenes de información de manera distribuida.

---

## ✅ Resultado Final

Se logró implementar exitosamente un análisis de Machine Learning en entorno Big Data utilizando PySpark MLlib, cumpliendo con los objetivos de la práctica.

In [10]:
# ============================================================
# PRÁCTICA: RDD EN PYSPARK (ESTADÍSTICAS DESCRIPTIVAS BÁSICAS)
# Fuente: NYC Taxi (Parquet) - Yellow Tripdata (2025-01..03)
# ============================================================

import os, sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===============================
# 0) SPARK SESSION (tu configuración)
# ===============================
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PRACTICA_ML_PYSPARK")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
    .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
    # estabilidad
    .config("spark.python.worker.faulthandler.enabled", "true")
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print("Spark listo ✅")
print("Python:", sys.executable)

# ===============================
# 1) CARGA DE LOS ARCHIVOS PARQUET
# ===============================
ruta_base = r"C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\6 - DATOS MASIVOS"

df = spark.read.parquet(
    f"{ruta_base}/yellow_tripdata_2025-01.parquet",
    f"{ruta_base}/yellow_tripdata_2025-02.parquet",
    f"{ruta_base}/yellow_tripdata_2025-03.parquet"
)

print("DataFrame cargado correctamente ✅")
print("Columnas disponibles:", df.columns)

# ============================================================
# CAMBIO PRINCIPAL (MLlib): Mejorar FEATURES + Random Forest
# - Antes: LinearRegression con 2 features
# - Ahora: RandomForestRegressor con más variables
# ============================================================

# ===============================
# 2) LIMPIEZA / FILTRADO (evitar NULLs y valores inválidos)
# ===============================
# Nota: Filtramos NULLs en TODAS las columnas que usaremos como features + label
df_ml = (
    df
    .filter(F.col("trip_distance").isNotNull())
    .filter(F.col("passenger_count").isNotNull())
    .filter(F.col("payment_type").isNotNull())
    .filter(F.col("tip_amount").isNotNull())
    .filter(F.col("tolls_amount").isNotNull())
    .filter(F.col("extra").isNotNull())
    .filter(F.col("congestion_surcharge").isNotNull())
    .filter(F.col("fare_amount").isNotNull())
    .filter(F.col("total_amount").isNotNull())

    # Valores razonables
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") < 100)

    .filter(F.col("fare_amount") > 0)
    .filter(F.col("fare_amount") < 200)

    .filter(F.col("tip_amount") >= 0)
    .filter(F.col("tip_amount") < 200)

    .select(
        "trip_distance",
        "passenger_count",
        "payment_type",
        "tip_amount",
        "tolls_amount",
        "extra",
        "congestion_surcharge",
        "fare_amount"
    )
)

print("Registros para ML (sin NULLs):", df_ml.count())
df_ml.show(5)

# ===============================
# 3) LIBRERÍAS MLlib
# ===============================
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# ===============================
# 4) CREAR VECTOR DE FEATURES (más variables)
# ===============================
# CAMBIO: ahora usamos más columnas para explicar mejor la tarifa
feature_cols = [
    "trip_distance",
    "passenger_count",
    "payment_type",
    "tip_amount",
    "tolls_amount",
    "extra",
    "congestion_surcharge"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_vector = assembler.transform(df_ml)

df_vector.select("features", "fare_amount").show(5, truncate=False)

# ===============================
# 5) TRAIN / TEST SPLIT
# ===============================
train_data, test_data = df_vector.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_data.count())
print("Test:", test_data.count())

# ===============================
# 6) MODELO ML (Random Forest Regressor)
# ===============================
# CAMBIO: RandomForest captura relaciones NO lineales (mejor para este problema)
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=50,     # puedes subir a 100 si tu PC lo aguanta bien
    maxDepth=8,      # profundidad moderada (balance rendimiento/tiempo)
    seed=42
)

model = rf.fit(train_data)
print("Modelo Random Forest entrenado ✅")

# ===============================
# 7) IMPORTANCIA DE VARIABLES (extra útil para tu explicación)
# ===============================
# Esto te dice qué variables aportaron más a la predicción
importances = model.featureImportances

print("\nImportancia de variables (de mayor a menor):")
imp_list = sorted(zip(feature_cols, importances.toArray()), key=lambda x: x[1], reverse=True)
for col, imp in imp_list:
    print(f" - {col}: {imp:.6f}")

# ===============================
# 8) PREDICCIONES
# ===============================
predictions = model.transform(test_data)

predictions.select(
    "trip_distance",
    "passenger_count",
    "payment_type",
    "tip_amount",
    "tolls_amount",
    "extra",
    "congestion_surcharge",
    "fare_amount",
    "prediction"
).show(10)

# ===============================
# 9) EVALUACIÓN
# ===============================
evaluator_rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("\nRESULTADOS DEL MODELO (Random Forest)")
print("RMSE:", rmse)
print("R2:", r2)


# ===============================
# 10) CERRAR SPARK
# ===============================
spark.stop()
print("\nSpark detenido ✅")

Spark listo ✅
Python: C:\Users\Oscar Ferreira\venv_pyspark\Scripts\python.exe
DataFrame cargado correctamente ✅
Columnas disponibles: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
Registros para ML (sin NULLs): 8642355
+-------------+---------------+------------+----------+------------+-----+--------------------+-----------+
|trip_distance|passenger_count|payment_type|tip_amount|tolls_amount|extra|congestion_surcharge|fare_amount|
+-------------+---------------+------------+----------+------------+-----+--------------------+-----------+
|          1.6|              1|           1|       3.0|         0.0|  3.5|                 2.5|       10.0|
|          0.5|              1|           1|  